# Phase 4 - Court Judgment Prediction (hybrid RNN + transformer Phase-1 model)

**Task:** predict a binary outcome (0/1) for each court case from its full text; scored by macro-F1.

**What you need on disk (nothing else - the model code is embedded below):**
1. your Phase-1 hybrid checkpoint `hybrid_final.pt` (must contain `model_state` and `config`)
2. the Phase-1 tokenizer pickle `legal_bpe_v3.pkl` (the `.pt` does **not** contain it)
3. a Hugging Face token (Kaggle secret `hf_token`)

**Approach (hierarchical, long-document):** a judgment is cut into `CH`-token chunks; up to `K_CHUNKS` of them are selected (`SELECT` = head / tail / headtail / spread); the hybrid encodes every chunk into a vector; a document-level aggregator (learned attention pooling, mean, or BiGRU) combines the chunk vectors; a linear layer gives the outcome. Class-weighted loss for the imbalance; an optional dev-tuned threshold as calibration.

**Experiments:** full fine-tuning, partial (layer-wise) fine-tuning, LoRA, **LoRMA+** (compulsory), bottleneck adapters, knowledge distillation (student = hybrid with one GRU block dropped per group), plus long-document ablations (which chunks, which aggregator). AdaLoRA is **not** included.

> The course requires all **four** Phase-1 architectures and >= 6 substantially different experiments (at least 5 different adaptation techniques, LoRMA and knowledge distillation among them). This notebook covers the hybrid model; hyperparameters are placeholders - choosing and justifying them is your job (course rules).

## 0. Config - edit the paths

In [ ]:
TOKENIZER_PATH = "/kaggle/input/YOUR_DATASET/legal_bpe_v3.pkl"      # <-- Phase-1 tokenizer
CKPT_PATH      = "/kaggle/input/YOUR_DATASET/hybrid_final.pt"       # <-- Phase-1 hybrid checkpoint
HF_REPO = "Exploration-Lab/CS-781-Capstone"
OUT = "/kaggle/working"

CH         = 512      # tokens per chunk (<= the hybrid's max_seq_len of 1024)
STRIDE_CH  = 512      # chunk stride; < CH gives overlapping chunks
K_CHUNKS   = 4        # at most this many chunks of a document go through the model (cost is linear in this; 8 was ~2x slower)
SELECT     = "headtail"   # which chunks: head | tail | headtail | spread   (the long-document / section experiment)
CAP_CHARS  = 60000    # very long judgments: keep this many characters from each end (bounds tokenisation time)

## 1. Imports and Hugging Face login

In [ ]:
import copy, json, math, os, pickle, random, re, time, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| torch", torch.__version__)

from collections import Counter

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("hf_token")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")        # Colab / local: export HF_TOKEN yourself
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN); print("HF login ok")
else:
    print("no HF token found - dataset download may fail")

## 2. Phase-1 tokenizer (class copied from your Phase-1 notebook)

In [ ]:
class LegalBPETokenizer:
    """Phase-1 tokenizer, copied unchanged from your Phase-1 notebook (needs legal_bpe_v3.pkl)."""

    def __init__(self, model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])
        self.merge_rank = {tuple(pair): rank for rank, (pair, _) in enumerate(self.merges)}
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]
        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")
        self._special_re = (re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
                            if self.special_tokens else None)
        self._cache = {}

    def _encode_piece(self, piece_bytes):
        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached
        ids = list(piece_bytes)
        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break
            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):
        if self.lowercase:
            text = text.lower()
        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):
        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])
        return b"".join(chunks).decode("utf-8", errors="replace")


tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
print("vocab size:", tokenizer.vocab_size, "| lowercase:", tokenizer.lowercase, "| eos id:", tokenizer.eos_id)
print([tokenizer.decode([i]) for i in tokenizer.encode("The appellant filed this appeal.")])

## 3. Phase-1 hybrid code (copied unchanged) and checkpoint loading

In [ ]:
# ---- attention_ed.py pieces used by the hybrid (Phase 1, unchanged) ----
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        x_f = x.float()
        x_f = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x_f * self.weight).to(x.dtype)


def build_rope_cache(max_len: int, d_head: int, base: float = 10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
    angles = torch.outer(torch.arange(max_len, dtype=torch.float32), inv_freq)
    return torch.cos(angles), torch.sin(angles)


def apply_rope(x, cos, sin):
    x_f = x.float()
    x1, x2 = x_f[..., 0::2], x_f[..., 1::2]
    cos, sin = cos[None, None, :, :], sin[None, None, :, :]
    out = torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1).flatten(-2)
    return out.to(x.dtype)


# ---- hybrid_layers.py (Phase 1, unchanged) ----
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)


class FastRecurrentBlock(nn.Module):
    """Pre-norm cuDNN GRU block (unidirectional) + FFN."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.rnn = nn.GRU(input_size=d_model, hidden_size=d_model, num_layers=1, batch_first=True)
        self.ln2 = RMSNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
    def forward(self, x):
        r, _ = self.rnn(self.ln1(x))
        x = x + r
        return x + self.ffn(self.ln2(x))


class FastAttentionBlock(nn.Module):
    """Causal transformer block with RoPE + QK-norm, fused scaled_dot_product_attention."""
    def __init__(self, d_model, n_heads, max_seq_len, d_ff, dropout, rope=True, qk_norm=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads, self.d_head = d_model, n_heads, d_model // n_heads
        self.ln1 = RMSNorm(d_model)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.qk_norm = qk_norm
        if qk_norm:
            self.q_norm = RMSNorm(self.d_head)
            self.k_norm = RMSNorm(self.d_head)
        self.rope = rope
        if rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        self.ln2 = RMSNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.dropout = dropout

    def apply_rope(self, x):
        return apply_rope(x, self.rope_cos[:x.shape[2]], self.rope_sin[:x.shape[2]])

    def forward(self, x):
        B, T, C = x.shape
        r = self.ln1(x)
        q = self.q_proj(r).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(r).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(r).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        if self.qk_norm:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rope:
            q, k = self.apply_rope(q), self.apply_rope(k)
        r = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                                           dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        r = self.out_proj(r.transpose(1, 2).contiguous().view(B, T, C))
        x = x + r
        return x + self.ffn(self.ln2(x))


# ---- model_arch_hybrid.py (Phase 1, unchanged) ----
from dataclasses import dataclass

@dataclass
class HybridConfig:
    vocab_size: int
    d_model: int = 768
    n_heads: int = 12
    n_recurrent_layers: int = 8
    n_attention_layers: int = 4
    d_ff: int = 2304
    max_seq_len: int = 1024
    dropout: float = 0.0
    rope: bool = True
    qk_norm: bool = True


class HybridLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        blocks = []
        for _ in range(4):                                   # [R R A] x 4
            blocks.append(FastRecurrentBlock(cfg.d_model, cfg.d_ff, cfg.dropout))
            blocks.append(FastRecurrentBlock(cfg.d_model, cfg.d_ff, cfg.dropout))
            blocks.append(FastAttentionBlock(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.d_ff, cfg.dropout,
                                             rope=cfg.rope, qk_norm=cfg.qk_norm))
        self.blocks = nn.ModuleList(blocks)
        self.ln_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.tok_emb(x)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_f(x))

In [ ]:
def load_lm(path=CKPT_PATH, dropout=0.1):
    # weights_only=False: this is YOUR OWN checkpoint (config dict + tensors), not an untrusted file
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = HybridConfig(**{k: v for k, v in ck["config"].items() if k in HybridConfig.__dataclass_fields__})
    assert cfg.vocab_size == tokenizer.vocab_size, \
        f"checkpoint vocab {cfg.vocab_size} != tokenizer vocab {tokenizer.vocab_size}: wrong tokenizer .pkl?"
    cfg.dropout = dropout                                    # Phase 1 used 0.0; a little helps fine-tuning
    lm = HybridLM(cfg)
    lm.load_state_dict(ck["model_state"])                    # rope caches are non-persistent buffers: rebuilt, not loaded
    return lm

_lm = load_lm(dropout=0.0)
print(_lm.cfg)
print(f"loaded Phase-1 hybrid: {sum(p.numel() for p in _lm.parameters())/1e6:.1f}M params, {len(_lm.blocks)} blocks")
assert _lm.cfg.max_seq_len >= CH, "CH must be <= max_seq_len"

## 4. Data
### 4.1 Peek at the raw JSON (verify the layout and the label key before trusting the parser)

In [ ]:
def hf_json(path):
    return json.load(open(hf_hub_download(HF_REPO, path, repo_type="dataset"), encoding="utf-8"))

for name, path in [("train", "Phase_4_CJPE/Train/train.json"), ("dev", "Phase_4_CJPE/Train/dev.json"),
                   ("test", "Phase_4_CJPE/Test/test.json")]:
    raw = hf_json(path)
    first = next(iter(raw.items())) if isinstance(raw, dict) else raw[0]
    if isinstance(first, dict):
        show = {k: (str(v)[:120] + "..." if len(str(v)) > 120 else v) for k, v in first.items()}
    else:
        show = str(first)[:300]
    print(name, type(raw).__name__, "len", len(raw), "\n   first:", show, "\n")

### 4.2 Parse train / dev / test, class balance

In [ ]:
LABEL_KEYS = ("label", "labels", "y", "outcome", "target", "judgment", "decision")

def parse_docs(raw):
    if isinstance(raw, dict):
        items = [(k, v) for k, v in raw.items()]
    else:
        items = [(d["id"], d) for d in raw]
    docs = []
    for did, d in items:
        if not isinstance(d, dict):
            d = {"text": d}
        lab = next((d[k] for k in LABEL_KEYS if k in d), None)
        docs.append(dict(id=d.get("id", did), text=d["text"], label=None if lab is None else int(lab)))
    return docs

train_docs = parse_docs(hf_json("Phase_4_CJPE/Train/train.json"))
dev_docs   = parse_docs(hf_json("Phase_4_CJPE/Train/dev.json"))
test_docs  = parse_docs(hf_json("Phase_4_CJPE/Test/test.json"))
for name, ds in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    labs = [d["label"] for d in ds]
    extra = f" | label counts {dict(Counter(labs))}" if labs[0] is not None else ""
    print(f"{name}: {len(ds)} docs{extra}")
assert all(d["label"] in (0, 1) for d in train_docs + dev_docs), "labels must be 0/1 - check the label key printed above"
assert len({str(d["id"]) for d in test_docs}) == len(test_docs), "duplicate test ids"
print("median chars/doc:", int(np.median([len(d["text"]) for d in train_docs])),
      "| max:", max(len(d["text"]) for d in train_docs))

### 4.3 Tokenise and chunk

In [ ]:
def chunk_ids(ids, ch=CH, stride=STRIDE_CH):
    starts = [0]
    while starts[-1] + ch < len(ids):
        starts.append(starts[-1] + stride)
    return [ids[s:s + ch] for s in starts]

def tokenise_doc(text):
    if len(text) > 2 * CAP_CHARS:
        text = text[:CAP_CHARS] + " " + text[-CAP_CHARS:]
    ids = tokenizer.encode(text)
    return ids if ids else [tokenizer.eos_id]

t0 = time.time()
for d in train_docs + dev_docs + test_docs:
    ids = tokenise_doc(d["text"])
    d["n_tok"] = len(ids)
    d["chunks"] = chunk_ids(ids)
print(f"tokenised in {time.time()-t0:.0f}s | tokens/doc: median {int(np.median([d['n_tok'] for d in train_docs]))}, "
      f"p90 {int(np.percentile([d['n_tok'] for d in train_docs], 90))} | chunks/doc: "
      f"median {int(np.median([len(d['chunks']) for d in train_docs]))}, max {max(len(d['chunks']) for d in train_docs)}")

### 4.4 Chunk selection and batching

In [ ]:
def select_idx(n, select=None, K=None):
    """Which of the n chunks of a document are used.  head = first K, tail = last K (judgments end with the outcome),
    headtail = first K/2 + last K/2, spread = K evenly spaced chunks over the whole document."""
    select, K = select or SELECT, K or K_CHUNKS
    if n <= K:
        return list(range(n))
    if select == "head":
        return list(range(K))
    if select == "tail":
        return list(range(n - K, n))
    if select == "headtail":
        h = K // 2
        return list(range(h)) + list(range(n - (K - h), n))
    if select == "spread":
        return sorted({round(i * (n - 1) / max(1, K - 1)) for i in range(K)})
    raise ValueError(select)

def collate(docs, select=None, K=None):
    """-> X (N,T) chunk tokens | lens (N,) | cnt (D,) chunks per doc | Y (D,) labels.  Chunks of all docs are stacked."""
    seqs, cnt = [], []
    for d in docs:
        idx = select_idx(len(d["chunks"]), select, K)
        seqs += [d["chunks"][i] for i in idx]; cnt.append(len(idx))
    T = max(len(s) for s in seqs)
    X = torch.zeros(len(seqs), T, dtype=torch.long)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s)
    lens = torch.tensor([len(s) for s in seqs])
    Y = torch.tensor([d["label"] if d["label"] is not None else -1 for d in docs])
    return X, lens, torch.tensor(cnt), Y

print("example selections for a 20-chunk doc:", {s: select_idx(20, s) for s in ("head", "tail", "headtail", "spread")})

### 4.5 Checkpoint / tokenizer sanity check

In [ ]:
# Does checkpoint + tokenizer agree? LM loss on real legal text should be ~3-4 nats; ~10 means a wrong tokenizer pkl.
x = torch.tensor([train_docs[0]["chunks"][0]]); _lm.eval()
with torch.no_grad():
    lg = _lm(x)
print("Phase-1 LM loss on a train chunk:", round(F.cross_entropy(lg[0, :-1], x[0, 1:]).item(), 3), "| tokens:", x.size(1))
del _lm

## 5. Metric (macro-F1) and threshold calibration

In [ ]:
def binary_scores(gold, pred):
    """Macro precision / recall / F1 over the two classes (the competition metric is macro-F1)."""
    P, R, F1, per = [], [], [], {}
    for c in (0, 1):
        tp = sum(1 for g, p in zip(gold, pred) if g == c and p == c)
        npred = sum(1 for p in pred if p == c); ngold = sum(1 for g in gold if g == c)
        p = tp / npred if npred else 0.0; r = tp / ngold if ngold else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        P.append(p); R.append(r); F1.append(f); per[c] = f
    acc = sum(1 for g, p in zip(gold, pred) if g == p) / max(1, len(gold))
    return dict(macro_f1=sum(F1) / 2, macro_p=sum(P) / 2, macro_r=sum(R) / 2, f1_class0=per[0], f1_class1=per[1], acc=acc)

def best_threshold(probs, gold, grid=None):
    """Post-processing: decision threshold on P(class 1) that maximises dev macro-F1 (a calibration step - report it as one)."""
    grid = grid or [i / 100 for i in range(5, 96, 1)]
    best = max(grid, key=lambda t: binary_scores(gold, [int(p >= t) for p in probs])["macro_f1"])
    return best, binary_scores(gold, [int(p >= best) for p in probs])["macro_f1"]

## 6. Adaptation methods: LoRA, LoRMA+, bottleneck adapters

In [ ]:
class LoRALinear(nn.Module):
    """LoRA: W x + s * B A x with W frozen.  rs=True -> rank-stabilised scaling s = alpha / sqrt(r) (rsLoRA)."""
    def __init__(self, base, r=16, alpha=32, rs=False, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / math.sqrt(r) if rs else alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.T @ self.B.T) * self.scale

def apply_lora(module, r=16, alpha=32, rs=False):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRALinear(child, r, alpha, rs))
        else:
            apply_lora(child, r, alpha, rs)


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel, Modi 2025): multiplicative update  h = (I + (alpha/r) B A) W0 x ,  B=0 at init."""
    def __init__(self, base, r=8, alpha=8):
        super().__init__()
        self.base = base
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + self.scale * ((z @ self.A.T) @ self.B.T)
        return z + self.base.bias if self.base.bias is not None else z

def apply_lorma(module, targets=("q_proj", "v_proj"), r=8, alpha=8):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, LoRMALinear(child, r, alpha))
        else:
            apply_lorma(child, targets, r, alpha)



class AdapterBlock(nn.Module):
    """Bottleneck adapter (Houlsby-style) after a whole block: x + up(GELU(down(x))). `up` starts at zero = identity."""
    def __init__(self, block, d_model, bottleneck=64):
        super().__init__()
        self.block = block
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        x = self.block(x)
        return x + self.up(F.gelu(self.down(x)))

def apply_adapters(lm, bottleneck=64):
    lm.blocks = nn.ModuleList([AdapterBlock(b, lm.cfg.d_model, bottleneck) for b in lm.blocks])

def n_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

## 7. Model, `build_model`, distillation student

In [ ]:
from torch.utils.checkpoint import checkpoint

class CJModel(nn.Module):
    """Phase-1 hybrid (8 GRU + 4 attention blocks, causal) reads each chunk -> chunk vector [mean ; last-token state]
    -> document-level aggregation over the chunks (agg = 'attn' learned attention pooling | 'mean' | 'gru' BiGRU)
    -> 2-way outcome logits."""
    def __init__(self, lm, agg="attn", proj=512, rnn_hidden=256, dropout=0.1):
        super().__init__()
        self.lm, self.agg = lm, agg
        self.n_nograd = 0        # first n blocks (and the embedding) run under no_grad: no backward through them (set by train_fast)
        self.use_ckpt = False    # gradient checkpointing on the trainable blocks: less memory, ~30% slower
        d = lm.cfg.d_model
        self.proj = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d, proj), nn.GELU())
        if agg == "attn":
            self.att = nn.Linear(proj, 1); zdim = proj
        elif agg == "gru":
            self.rnn = nn.GRU(proj, rnn_hidden, batch_first=True, bidirectional=True); zdim = 2 * rnn_hidden
        elif agg == "mean":
            zdim = proj
        else:
            raise ValueError(agg)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(zdim, 2)

    def head_params(self):
        mods = [self.proj, self.out] + [getattr(self, n) for n in ("att", "rnn") if hasattr(self, n)]
        return [p for m in mods for p in m.parameters()]

    def features(self, x):
        nf = self.n_nograd
        if nf:
            with torch.no_grad():
                h = self.lm.tok_emb(x)
                for blk in self.lm.blocks[:nf]:
                    h = blk(h)
        else:
            h = self.lm.tok_emb(x)
        for blk in self.lm.blocks[nf:]:
            if self.use_ckpt and self.training and torch.is_grad_enabled():
                h = checkpoint(blk, h, use_reentrant=False)
            else:
                h = blk(h)
        return self.lm.ln_f(h)

    def forward(self, X, lens, cnt):
        h = self.features(X)                                                   # (N, T, d)
        N, T, _ = h.shape
        lens = lens.to(h.device)
        m = (torch.arange(T, device=h.device)[None, :] < lens[:, None]).to(h.dtype)
        mean = (h * m.unsqueeze(-1)).sum(1) / lens.unsqueeze(-1).to(h.dtype)
        lastv = h[torch.arange(N, device=h.device), lens - 1]
        v = self.proj(torch.cat([mean, lastv], -1))                            # (N, proj): one vector per chunk
        D, Kmax = cnt.numel(), int(cnt.max())
        pad = v.new_zeros(D, Kmax, v.size(-1))
        mask = torch.zeros(D, Kmax, dtype=torch.bool, device=v.device)
        pos = 0
        for i, c in enumerate(cnt.tolist()):
            pad[i, :c] = v[pos:pos + c]; mask[i, :c] = True; pos += c
        if self.agg == "mean":
            z = (pad * mask.unsqueeze(-1).to(pad.dtype)).sum(1) / mask.sum(1, keepdim=True).to(pad.dtype)
        elif self.agg == "attn":
            s = self.att(pad).squeeze(-1).float().masked_fill(~mask, float("-inf"))
            w = F.softmax(s, dim=1).to(pad.dtype)
            z = (w.unsqueeze(-1) * pad).sum(1)
        else:
            packed = nn.utils.rnn.pack_padded_sequence(pad, cnt, batch_first=True, enforce_sorted=False)
            o, _ = self.rnn(packed)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=Kmax)
            z = (o * mask.unsqueeze(-1).to(o.dtype)).sum(1) / mask.sum(1, keepdim=True).to(o.dtype)
        return self.out(self.drop(z))                                          # (D, 2)


def build_model(method, agg="attn", r=16, alpha=32, lorma_r=8, lorma_alpha=8, bottleneck=64):
    lm = load_lm()
    if method not in ("full", "partial"):
        for p in lm.parameters(): p.requires_grad = False
    if method == "lora":      apply_lora(lm.blocks, r, alpha, rs=False)
    elif method == "lorma":   apply_lorma(lm.blocks, ("q_proj", "v_proj"), lorma_r, lorma_alpha)
    elif method == "adapters": apply_adapters(lm, bottleneck)
    elif method == "partial":                       # layer-wise: freeze embeddings + lower half of the blocks
        for p in lm.tok_emb.parameters(): p.requires_grad = False
        for b in lm.blocks[: len(lm.blocks) // 2]:
            for p in b.parameters(): p.requires_grad = False
    elif method != "full":
        raise ValueError(method)
    return CJModel(lm, agg).to(dev)


def make_student(teacher):
    """Smaller student: from each [R R A] group keep [R A] (drop the 2nd GRU block) -> 8 of 12 blocks, shared embeddings,
    initialised from the teacher incl. its trained head. The teacher must be the plain full-fine-tuned model."""
    st = copy.deepcopy(teacher)
    keep = [i for i in range(len(st.lm.blocks)) if i % 3 != 1]
    st.lm.blocks = nn.ModuleList([st.lm.blocks[i] for i in keep])
    for p in st.parameters(): p.requires_grad = True
    return st

## 8. Inference and evaluation

In [ ]:
@torch.no_grad()
def predict_probs(model, docs, select=None, K=None, bs=4):
    """-> list of P(class 1), aligned with `docs`."""
    model.eval()
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["chunks"]))
    probs = [0.0] * len(docs)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        X, lens, cnt, _ = collate([docs[j] for j in idx], select, K)
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            lg = model(X.to(dev), lens, cnt)
        p = F.softmax(lg.float(), -1)[:, 1].cpu().tolist()
        for j, pj in zip(idx, p):
            probs[j] = pj
    return probs

def evaluate(model, docs, select=None, K=None, threshold=0.5, verbose=True):
    probs = predict_probs(model, docs, select, K)
    res = binary_scores([d["label"] for d in docs], [int(p >= threshold) for p in probs])
    res["probs"] = probs
    if verbose:
        print(f"   dev macro-F1 {res['macro_f1']:.4f} (P {res['macro_p']:.4f} R {res['macro_r']:.4f}) | "
              f"F1 class0 {res['f1_class0']:.4f} class1 {res['f1_class1']:.4f} | acc {res['acc']:.4f}")
    return res

## 9. Training loop (all methods; best-epoch restore on dev)

In [ ]:
def train(method, train_docs, dev_docs, epochs=4, bs=2, accum=4, lr_bb=3e-5, lr_head=1e-3, agg="attn",
          teacher=None, kd_T=2.0, kd_alpha=0.5, cw_power=0.5, select=None, K=None, seed=0):
    """method: full | partial | lora | lorma | adapters | kd.   Returns (model restored to its best dev epoch, best dev macro-F1).
    bs = documents per step; each document sends up to K chunks through the model, so lower bs if you run out of memory."""
    random.seed(seed); torch.manual_seed(seed)
    if method == "kd":
        assert teacher is not None, "pass teacher=<your fully fine-tuned model>"
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
        model = make_student(teacher).to(dev)
    else:
        model = build_model(method, agg)
    print(f"[{method}/{agg}/{select or SELECT}] params total {n_params(model)/1e6:.1f}M | trainable {n_params(model, True)/1e6:.2f}M")

    head_ids = {id(p) for p in model.head_params()}
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    hd = [p for p in model.parameters() if id(p) in head_ids]
    groups, base_lrs = [], []
    if bb: groups.append({"params": bb, "lr": lr_bb}); base_lrs.append(lr_bb)
    groups.append({"params": hd, "lr": lr_head}); base_lrs.append(lr_head)
    opt = torch.optim.AdamW(groups, weight_decay=0.01)

    steps_total = epochs * math.ceil(math.ceil(len(train_docs) / bs) / accum)
    warm = max(1, int(0.06 * steps_total))
    def lr_mult(s):
        return s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps_total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    cnt = torch.tensor([sum(d["label"] == c for d in train_docs) for c in (0, 1)], dtype=torch.float) + 1
    cw = ((cnt.sum() / cnt / 2) ** cw_power).to(dev) if cw_power else torch.ones(2, device=dev)
    cw = cw / cw.mean()                                          # class-balanced loss (cw_power=0 turns it off)
    print("class weights:", [round(x, 3) for x in cw.tolist()])

    best, best_state, step = -1.0, None, 0
    for ep in range(epochs):
        model.train(); t0 = time.time(); run = 0.0
        order = list(range(len(train_docs))); random.shuffle(order)
        batches = [order[i:i + bs] for i in range(0, len(order), bs)]
        for bi, b in enumerate(batches):
            X, lens, c, Y = collate([train_docs[i] for i in b], select, K)
            X, Y = X.to(dev), Y.to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                lg = model(X, lens, c)
            loss = F.cross_entropy(lg.float(), Y, weight=cw)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                    tl = teacher(X, lens, c)
                kl = F.kl_div(F.log_softmax(lg.float() / kd_T, -1), F.softmax(tl.float() / kd_T, -1),
                              reduction="batchmean") * kd_T ** 2
                loss = kd_alpha * loss + (1 - kd_alpha) * kl
            scaler.scale(loss / accum).backward(); run += loss.item()
            if (bi + 1) % accum == 0 or bi + 1 == len(batches):
                step += 1
                for g, b0 in zip(opt.param_groups, base_lrs): g["lr"] = b0 * lr_mult(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        print(f"ep {ep+1}/{epochs} loss {run/max(1, len(batches)):.4f} ({time.time()-t0:.0f}s)")
        if dev_docs:
            sc = evaluate(model, dev_docs, select, K)
            if sc["macro_f1"] > best:
                best = sc["macro_f1"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state); print(f"restored best epoch, dev macro-F1 {best:.4f}")
    return model, best

## 9b. FAST time-budgeted trainer (use this one)
Trains for a fixed number of minutes instead of fixed epochs, optionally only the top blocks, so every run has a predictable cost. It prints docs/s so you can see the real speed.

In [ ]:
def train_fast(method, train_docs, dev_docs, minutes=40, K=None, bs=2, accum=4, lr_bb=3e-5, lr_head=1e-3, agg="attn",
               teacher=None, kd_T=2.0, kd_alpha=0.5, cw_power=0.5, select=None, top_blocks=None, ckpt=False, seed=0):
    """Time-budgeted trainer: trains for `minutes` of wall-clock time (random documents, schedule follows elapsed time), then
    evaluates on dev once.  method: full | partial | lora | lorma | adapters | kd.
    top_blocks=n  -> only the top n blocks (+ head) are trained; the embedding and lower blocks run under no_grad (no backward
                     through them).  n=4 is one [R R A] group; n=0 trains the head only (backbone forward only: the cheapest run).
    ckpt=True     -> gradient checkpointing (only if you run out of GPU memory; ~30% slower).
    K             -> chunks per document (default K_CHUNKS).  Cost per document is linear in K."""
    random.seed(seed); torch.manual_seed(seed)
    if method == "kd":
        assert teacher is not None, "pass teacher=<your fully fine-tuned model>"
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
        model = make_student(teacher).to(dev)
    else:
        model = build_model(method, agg)
    model.use_ckpt = ckpt
    if top_blocks is not None and method in ("full", "partial", "kd"):
        nf = len(model.lm.blocks) - top_blocks
        for p in model.lm.tok_emb.parameters(): p.requires_grad = False
        for b in model.lm.blocks[:nf]:
            for p in b.parameters(): p.requires_grad = False
        model.n_nograd = nf
    print(f"[{method}/{agg}/{select or SELECT}/K={K or K_CHUNKS}/top={top_blocks}] params {n_params(model)/1e6:.1f}M | "
          f"trainable {n_params(model, True)/1e6:.2f}M | budget {minutes} min")

    head_ids = {id(p) for p in model.head_params()}
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    hd = [p for p in model.parameters() if id(p) in head_ids]
    groups, base_lrs = [], []
    if bb: groups.append({"params": bb, "lr": lr_bb}); base_lrs.append(lr_bb)
    groups.append({"params": hd, "lr": lr_head}); base_lrs.append(lr_head)
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    cnt0 = torch.tensor([sum(d["label"] == c for d in train_docs) for c in (0, 1)], dtype=torch.float) + 1
    cw = ((cnt0.sum() / cnt0 / 2) ** cw_power).to(dev) if cw_power else torch.ones(2, device=dev)
    cw = cw / cw.mean()

    budget, t0, mb, seen, run, nrun, last_log = minutes * 60.0, time.time(), 0, 0, 0.0, 0, time.time()
    done = False
    while not done:
        order = list(range(len(train_docs))); random.shuffle(order)
        for i in range(0, len(order), bs):
            el = time.time() - t0
            if el >= budget:
                done = True; break
            model.train()
            X, lens, c, Y = collate([train_docs[j] for j in order[i:i + bs]], select, K)
            X, Y = X.to(dev), Y.to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                lg = model(X, lens, c)
            loss = F.cross_entropy(lg.float(), Y, weight=cw)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                    tl = teacher(X, lens, c)
                kl = F.kl_div(F.log_softmax(lg.float() / kd_T, -1), F.softmax(tl.float() / kd_T, -1),
                              reduction="batchmean") * kd_T ** 2
                loss = kd_alpha * loss + (1 - kd_alpha) * kl
            scaler.scale(loss / accum).backward()
            run += loss.item(); nrun += 1; seen += len(Y); mb += 1
            if mb % accum == 0:
                prog = el / budget
                mult = prog / 0.05 if prog < 0.05 else 0.5 * (1 + math.cos(math.pi * (prog - 0.05) / 0.95))
                for g, b0 in zip(opt.param_groups, base_lrs): g["lr"] = b0 * max(mult, 0.02)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if time.time() - last_log > 300:
                print(f"   {(time.time()-t0)/60:5.1f} min | docs seen {seen} ({seen/len(train_docs):.2f} epochs) | "
                      f"{seen/(time.time()-t0):.2f} docs/s | loss {run/max(1,nrun):.4f}")
                run, nrun, last_log = 0.0, 0, time.time()
    print(f"trained {(time.time()-t0)/60:.1f} min: {seen} docs = {seen/len(train_docs):.2f} epochs ({seen/(time.time()-t0):.2f} docs/s)")
    sc = evaluate(model, dev_docs, select, K) if dev_docs else {"macro_f1": float("nan")}
    return model, sc["macro_f1"]

## 10. Submission writer
`prediction.json` (a list of `{"id", "pred"}`) goes at the **root** of the zip. Upload each zip to Codabench.

In [ ]:
def make_submission(model, docs, tag, threshold=0.5, select=None, K=None):
    """prediction.json = [{"id": ..., "pred": 0/1}, ...], one entry per test case, zipped with the json at the zip root."""
    probs = predict_probs(model, docs, select, K)
    sub = [{"id": d["id"], "pred": int(p >= threshold)} for d, p in zip(docs, probs)]
    assert len(sub) == len(docs) and all(s["pred"] in (0, 1) for s in sub)
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(sub, f)
    name = f"{OUT}/submission_p4_{tag}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print(name, zipfile.ZipFile(name).namelist(), "| cases:", len(sub), "| predicted 1:", sum(s["pred"] for s in sub))
    return name

results = {}

## 11. Experiment 1 - full fine-tuning (also the KD teacher)
Everything trainable (backbone lr 3e-5, head lr 1e-3). Keep this model in memory for distillation. Judgments are long: if you run out of GPU memory, lower `bs` or `K_CHUNKS`.

In [ ]:
m_full, results["full"] = train_fast("full", train_docs, dev_docs, minutes=90, top_blocks=4)
torch.save(m_full.state_dict(), f"{OUT}/p4_hybrid_full.pt")
make_submission(m_full, test_docs, "hybrid_full")

## 12. Experiment 2 - partial (layer-wise) fine-tuning
Embeddings and the lower half of the blocks frozen; the upper half and the head train.

In [ ]:
m_part, results["partial"] = train_fast("partial", train_docs, dev_docs, minutes=45, top_blocks=4)   # same trained blocks, but only 45 min
make_submission(m_part, test_docs, "hybrid_partial")

## 13. Experiment 3 - LoRA
Frozen backbone, rank-16 adapters on every Linear layer (attention and FFN).

In [ ]:
m_lora, results["lora"] = train_fast("lora", train_docs, dev_docs, minutes=45, lr_bb=5e-4)
make_submission(m_lora, test_docs, "hybrid_lora")

## 14. Experiment 4 - LoRMA+ (compulsory)
Multiplicative low-rank adaptation (Bihany, Patel, Modi 2025) of the query/value projections of the 4 attention blocks.

In [ ]:
m_lorma, results["lorma"] = train_fast("lorma", train_docs, dev_docs, minutes=45, lr_bb=3e-4)
make_submission(m_lorma, test_docs, "hybrid_lorma")

## 15. Experiment 5 - bottleneck adapters
A small zero-initialised down/up adapter after every block; the backbone stays frozen.

In [ ]:
m_adapt, results["adapters"] = train_fast("adapters", train_docs, dev_docs, minutes=45, lr_bb=1e-3)
make_submission(m_adapt, test_docs, "hybrid_adapters")

## 16. Experiment 6 - knowledge distillation
Teacher = experiment 1. Student = teacher with the 2nd GRU block of every [R R A] group removed (8 of 12 blocks), initialised from the teacher, trained on hard labels + temperature-scaled teacher logits. Compare its parameter count, speed and F1 with the teacher.

In [ ]:
m_kd, results["kd"] = train_fast("kd", train_docs, dev_docs, minutes=45, teacher=m_full, lr_bb=5e-5, top_blocks=4)
make_submission(m_kd, test_docs, "hybrid_kd")

## 17. Long-document ablations (full fine-tuning; optional, each is a full training run)
Which part of a judgment carries the signal? Compare `head` (beginning), `tail` (end - where the outcome usually is) and `spread` against the default `headtail`; then the aggregator (`attn` is the default, `mean` and `gru` below). Report what the dev scores say about sections.

In [ ]:
for sel in ["head", "tail", "spread"]:
    m_tmp, results[f"full_{sel}"] = train_fast("full", train_docs, dev_docs, minutes=30, top_blocks=4, select=sel)
    del m_tmp; torch.cuda.empty_cache()
for ag in ["mean", "gru"]:
    m_tmp, results[f"full_agg_{ag}"] = train_fast("full", train_docs, dev_docs, minutes=30, top_blocks=4, agg=ag)
    del m_tmp; torch.cuda.empty_cache()

## 18. Summary, threshold calibration and error-analysis starters

In [ ]:
print("dev macro-F1:")
for k, v in results.items():
    print(f"  {k:20s} {v:.4f}")

res = evaluate(m_full, dev_docs, verbose=False)
probs, gold = res["probs"], [d["label"] for d in dev_docs]
t, f = best_threshold(probs, gold)
print(f"\nfull fine-tune: macro-F1 at threshold 0.5 = {res['macro_f1']:.4f} | best dev threshold {t:.2f} -> {f:.4f}")
print("   (a tuned threshold is fitted on dev - report it as post-processing, and expect a smaller gain on the test set)")

pred = [int(p >= 0.5) for p in probs]
fp = [d for d, p, g in zip(dev_docs, pred, gold) if p == 1 and g == 0]
fn = [d for d, p, g in zip(dev_docs, pred, gold) if p == 0 and g == 1]
ok = [d for d, p, g in zip(dev_docs, pred, gold) if p == g]
print(f"false positives {len(fp)} | false negatives {len(fn)} | correct {len(ok)}")
for name, grp in [("FP", fp), ("FN", fn), ("correct", ok)]:
    if grp: print(f"  {name:8s} median tokens/doc {int(np.median([d['n_tok'] for d in grp]))}")

## 19. (Optional) retrain on train + dev
Once you know the best method, epoch count and (if used) threshold, retrain on all labelled data (no dev set, so no best-epoch restore) and submit again.
```python
m_final, _ = train_fast("full", train_docs + dev_docs, None, minutes=<same as the best run>, top_blocks=4)
make_submission(m_final, test_docs, "hybrid_final", threshold=0.5)
```